# TC3-40 | Catálogo de Variáveis e Fontes de Dados Adicionais

**Responsável:** Sonia Andreia Ribeiro

**Sprint:** Fase 2

**Jira:** TC3-40

---

## Objetivo

Documentar as fontes de dados utilizadas no projeto, descrevendo suas variáveis, tipos de dados, chaves de integração e potencial de utilização na modelagem supervisionada.

Este catálogo servirá como referência para as atividades de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 160)


def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )


PROJECT_ROOT = encontrar_raiz_projeto()
DADOS_PRINCIPAL = PROJECT_ROOT / "dados" / "principal"
DADOS_COMPLEMENTAR = PROJECT_ROOT / "dados" / "complementar"
DATA_DICTIONARY_DIR = PROJECT_ROOT / "docs" / "data_dictionary"
DATA_DICTIONARY_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", PROJECT_ROOT)


# 1. Fontes de Dados

O projeto utiliza uma Feature Table principal (`ml_aluno`) e duas bases adicionais previamente aprovadas pelo grupo para enriquecimento analítico.

As bases adicionais serão integradas em etapa posterior, preservando a rastreabilidade da camada Gold.

In [ ]:
fontes = pd.DataFrame({
    "Fonte": [
        "ml_aluno",
        "Censo Escolar 2023",
        "INSE 2023"
    ],
    "Categoria": [
        "Camada Gold",
        "Educacional",
        "Socioeconômica"
    ],
    "Finalidade": [
        "Base principal para modelagem supervisionada",
        "Características das escolas e infraestrutura educacional",
        "Indicadores de nível socioeconômico dos estudantes"
    ]
})

fontes

# 2. Catálogo de Variáveis

Nesta etapa serão catalogadas as variáveis disponíveis em cada fonte de dados.

O objetivo é documentar a função de cada atributo, sem definir sua utilização na modelagem.

In [ ]:
import zipfile

zip_path = DADOS_PRINCIPAL / "ml_aluno.zip"
extract_path = DADOS_PRINCIPAL
ml_aluno_path = extract_path / "ml_aluno"

if not ml_aluno_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

arquivos = sorted(ml_aluno_path.rglob("*.parquet"))

print(f"Arquivos encontrados: {len(arquivos)}")

df = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos],
    ignore_index=True
)

print(f"Registros: {len(df):,}")
print(f"Variáveis: {df.shape[1]}")

df.head()


In [ ]:
# Recria o catálogo exatamente a partir do df
catalogo_gold = pd.DataFrame({
    "Variável": list(df.columns),
    "Tipo": df.dtypes.astype(str).tolist()
})

catalogo_gold["Fonte"] = "ml_aluno"
catalogo_gold["Descrição"] = ""
catalogo_gold["Categoria"] = ""

catalogo_gold

# 2. Catálogo de Variáveis da Feature Table `ml_aluno`

Nesta etapa é gerado o dicionário de dados da Feature Table `ml_aluno`, identificando o nome de cada variável, seu tipo de dado, sua descrição funcional e sua categoria dentro do projeto.

Este catálogo tem como objetivo padronizar o entendimento das variáveis e servir como referência para as etapas de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

In [ ]:
descricoes = {
    "id_aluno": ("Identificador único do aluno", "Identificador"),
    "id_escola": ("Identificador único da escola", "Identificador"),
    "id_municipio": ("Código oficial do município (IBGE)", "Territorial"),
    "nome_municipio": ("Nome do município", "Territorial"),
    "sigla_uf": ("Sigla da Unidade Federativa", "Territorial"),
    "ano_referencia": ("Ano de referência da avaliação", "Temporal"),
    "serie": ("Série escolar do estudante", "Acadêmica"),
    "rede": ("Código da rede de ensino", "Administrativa"),
    "rede_descricao": ("Descrição da rede de ensino", "Administrativa"),
    "caderno": ("Código do caderno da avaliação", "Avaliação"),
    "presenca": ("Indica a presença do aluno na avaliação", "Avaliação"),
    "preenchimento_caderno": ("Situação de preenchimento do caderno", "Avaliação"),
    "proficiencia": ("Proficiência obtida pelo aluno", "Desempenho"),
    "peso_aluno": ("Peso amostral atribuído ao aluno", "Amostragem"),
    "taxa_alfabetizacao_municipio_ano_anterior": ("Taxa municipal de alfabetização do ano anterior", "Indicador Educacional"),
    "media_portugues_municipio_ano_anterior": ("Média municipal de Língua Portuguesa do ano anterior", "Indicador Educacional"),
    "dataset_split": ("Conjunto destinado ao treino, teste ou validação", "Modelagem"),
    "alfabetizado": ("Classificação do estudante quanto à alfabetização", "Target"),
    "processed_at": ("Data/hora de processamento da Feature Table", "Controle Técnico"),
}

len(descricoes)


In [ ]:
catalogo_gold["Descrição"] = catalogo_gold["Variável"].apply(
    lambda x: descricoes.get(x, ("Pendente", ""))[0]
)

catalogo_gold["Categoria"] = catalogo_gold["Variável"].apply(
    lambda x: descricoes.get(x, ("", "Revisar"))[1]
)

catalogo_ml_aluno_path = DATA_DICTIONARY_DIR / "ml_aluno_catalogo_variaveis.csv"
catalogo_gold.to_csv(catalogo_ml_aluno_path, index=False, encoding="utf-8")

print(f"Catálogo ml_aluno salvo em: {catalogo_ml_aluno_path}")
catalogo_gold


# 3. Catálogo de Variáveis — Censo Escolar 2023

O Censo Escolar 2023 é uma das bases adicionais aprovadas pelo grupo para enriquecimento da Feature Table `ml_aluno`.

Nesta seção será realizado o inventário das variáveis disponíveis, identificando seus tipos de dados e preparando o dicionário técnico que servirá de apoio para a Engenharia de Features (TC3-41).

In [ ]:
for arq in sorted(DADOS_COMPLEMENTAR.iterdir()):
    print(arq.name)


In [ ]:
import zipfile

zip_censo = DADOS_COMPLEMENTAR / "microdados_ed_basica_2023.zip"
pasta_censo = DADOS_COMPLEMENTAR / "censo_escolar_2023"

if not pasta_censo.exists():
    with zipfile.ZipFile(zip_censo, "r") as zip_ref:
        zip_ref.extractall(pasta_censo)

print("Extração concluída!")


In [ ]:
# Localizar todos os arquivos extraídos
for arq in sorted(pasta_censo.rglob("*")):
    if arq.is_file():
        print(arq)

In [ ]:
df_censo = pd.read_csv(
    pasta_censo / "microdados_ed_basica_2023.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

print(f"Registros: {len(df_censo):,}")
print(f"Variáveis: {df_censo.shape[1]}")

df_censo.head()


## 3.1 Inventário das Variáveis do Censo Escolar

O inventário abaixo identifica todas as variáveis presentes na base do Censo Escolar 2023, registrando seus tipos de dados. As descrições funcionais serão incorporadas a partir do dicionário oficial do INEP.

In [ ]:
catalogo_censo = pd.DataFrame({
    "Variável": df_censo.columns,
    "Tipo": df_censo.dtypes.astype(str).values
})

catalogo_censo["Fonte"] = "Censo Escolar 2023"
catalogo_censo["Descrição"] = ""
catalogo_censo["Categoria"] = ""

print(f"Total de variáveis catalogadas: {len(catalogo_censo)}")

catalogo_censo.head(10)

## 3.2 Dicionário Oficial do Censo Escolar

O Censo Escolar disponibiliza um dicionário oficial contendo a descrição funcional de cada variável.

Nesta etapa, esse documento será utilizado para complementar automaticamente o catálogo técnico da base.

In [ ]:
arquivo_dic = DADOS_COMPLEMENTAR / "dicionário_dados_educação_básica.xlsx"

xls = pd.ExcelFile(arquivo_dic)

print("Abas disponíveis:")
print(xls.sheet_names)


## 3.3 Integração com o Dicionário Oficial do INEP

Para garantir consistência e rastreabilidade, as descrições das variáveis do Censo Escolar serão obtidas diretamente do dicionário oficial disponibilizado pelo INEP.

Esse processo evita documentação manual e mantém o catálogo reproduzível.

In [ ]:
# Ler a aba principal do dicionário do INEP
dic_censo = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta"
)

print(dic_censo.shape)
dic_censo.head()

In [ ]:
dic_preview = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta",
    header=None
)

dic_preview.head(15)

*Leitura estruturada do Dicionário do INEP*

Após inspecionar o arquivo, identificou-se que o cabeçalho do dicionário inicia na sétima linha da planilha. A leitura abaixo utiliza esse cabeçalho para estruturar corretamente os metadados das variáveis.

In [ ]:
dic_censo = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta",
    header=6
)

print(dic_censo.shape)
dic_censo.head()

In [ ]:
catalogo_censo = pd.DataFrame({
    "Variável": df_censo.columns,
    "Tipo Base": df_censo.dtypes.astype(str).values
})

catalogo_censo = catalogo_censo.merge(
    dic_censo[["Nome da Variável", "Descrição da Variável", "Tipo", "Categoria"]],
    left_on="Variável",
    right_on="Nome da Variável",
    how="left"
)

catalogo_censo = catalogo_censo.rename(columns={
    "Tipo": "Tipo Pandas"
})

catalogo_censo = catalogo_censo.drop(columns="Nome da Variável")

catalogo_censo.head(10)

# 4. Catálogo de Variáveis — INSE 2023

O Indicador de Nível Socioeconômico (INSE) representa a segunda base adicional aprovada pelo grupo para enriquecimento da Feature Table `ml_aluno`.

Nesta seção será realizado o inventário das variáveis da base de escolas, identificando seus tipos de dados e preparando o catálogo técnico para futura integração.

### 4.1 Critério de seleção da base INSE

O INSE 2023 é disponibilizado pelo INEP em quatro níveis de agregação: Brasil, estados, municípios e escolas.

Para este projeto foi selecionada exclusivamente a base **INSE_2023_escolas**, pois a Feature Table `ml_aluno` possui identificação da escola (`id_escola`), permitindo a integração em nível escolar.

As versões agregadas (Brasil, estados e municípios) não serão utilizadas, uma vez que representam indicadores resumidos e não oferecem o mesmo nível de granularidade necessário para o enriquecimento dos registros individuais dos alunos.

In [ ]:
df_inse = pd.read_excel(DADOS_COMPLEMENTAR / "INSE_2023_escolas.xlsx")

print(f"Registros: {len(df_inse):,}")
print(f"Variáveis: {df_inse.shape[1]}")

df_inse.head()


## 4.2 Inventário das Variáveis

Após a leitura da base INSE 2023, é realizado o inventário automático das variáveis, registrando seus tipos de dados e preparando o catálogo técnico para a etapa de enriquecimento da Feature Table `ml_aluno`.

In [ ]:
catalogo_inse = pd.DataFrame({
    "Variável": df_inse.columns,
    "Tipo Pandas": df_inse.dtypes.astype(str).values
})

catalogo_inse["Fonte"] = "INSE 2023"
catalogo_inse["Descrição"] = ""
catalogo_inse["Categoria"] = ""

catalogo_inse.head(10)

In [ ]:
df_inse = pd.read_excel(DADOS_COMPLEMENTAR / "INSE_2023_escolas.xlsx")

print(f"Registros: {len(df_inse):,}")
print(f"Variáveis: {df_inse.shape[1]}")

df_inse.head()


### 4.3 Catálogo das Variáveis do INSE

O catálogo abaixo documenta as principais variáveis da base INSE 2023, registrando seu tipo de dado, finalidade e categoria funcional.

Este dicionário será utilizado como referência para a etapa de Engenharia de Features (TC3-41).

In [ ]:
descricoes_inse = {
    "NU_ANO_SAEB": ("Ano de referência do SAEB", "Temporal"),
    "CO_UF": ("Código da Unidade Federativa", "Territorial"),
    "SG_UF": ("Sigla da Unidade Federativa", "Territorial"),
    "NO_UF": ("Nome da Unidade Federativa", "Territorial"),
    "CO_MUNICIPIO": ("Código oficial do município (IBGE)", "Territorial"),
    "NO_MUNICIPIO": ("Nome do município", "Territorial"),
    "ID_ESCOLA": ("Código identificador da escola", "Identificador"),
    "NO_ESCOLA": ("Nome da escola", "Identificador"),
    "TP_TIPO_REDE": ("Código do tipo de rede de ensino", "Administrativa"),
    "TP_LOCALIZACAO": ("Código da localização da escola", "Administrativa"),
    "MEDIA_INSE": ("Média do Indicador de Nível Socioeconômico", "Indicador Socioeconômico"),
    "INSE_CLASSIFICACAO": ("Classificação do nível socioeconômico", "Indicador Socioeconômico"),
    "PC_NIVEL_1": ("Percentual de estudantes no Nível I", "Distribuição INSE"),
    "PC_NIVEL_2": ("Percentual de estudantes no Nível II", "Distribuição INSE"),
    "PC_NIVEL_3": ("Percentual de estudantes no Nível III", "Distribuição INSE"),
    "PC_NIVEL_4": ("Percentual de estudantes no Nível IV", "Distribuição INSE"),
    "PC_NIVEL_5": ("Percentual de estudantes no Nível V", "Distribuição INSE"),
    "PC_NIVEL_6": ("Percentual de estudantes no Nível VI", "Distribuição INSE"),
    "PC_NIVEL_7": ("Percentual de estudantes no Nível VII", "Distribuição INSE"),
    "PC_NIVEL_8": ("Percentual de estudantes no Nível VIII", "Distribuição INSE"),
    "PC_NIVEL_9": ("Percentual de estudantes no Nível IX", "Distribuição INSE"),
    "PC_NIVEL_10": ("Percentual de estudantes no Nível X", "Distribuição INSE"),
    "TP_CAPITAL": (
        "Indicador se o município pertence a uma capital",
        "Territorial"
    ),
    "QT_ALUNOS_INSE": (
        "Quantidade de alunos considerados no cálculo do INSE",
        "Indicador Socioeconômico"
    )
}


catalogo_inse["Descrição"] = catalogo_inse["Variável"].map(
    lambda x: descricoes_inse.get(x, ("Pendente de documentação", "Revisar"))[0]
)

catalogo_inse["Categoria"] = catalogo_inse["Variável"].map(
    lambda x: descricoes_inse.get(x, ("", "Revisar"))[1]
)

catalogo_inse

# 5. Chaves de Integração

As fontes de dados utilizadas no projeto apresentam diferentes nomenclaturas para identificar escolas e municípios.

Esta seção documenta as chaves que serão utilizadas posteriormente no Notebook de Enriquecimento, garantindo a rastreabilidade entre a Feature Table `ml_aluno` e as bases adicionais.

In [ ]:
chaves_integracao = pd.DataFrame({
    "Base de Dados": [
        "ml_aluno",
        "Censo Escolar 2023",
        "INSE 2023"
    ],
    "Chave Escola": [
        "id_escola",
        "CO_ENTIDADE",
        "ID_ESCOLA"
    ],
    "Chave Município": [
        "id_municipio",
        "CO_MUNICIPIO",
        "CO_MUNICIPIO"
    ],
    "Granularidade": [
        "Aluno",
        "Escola",
        "Escola"
    ]
})

chaves_integracao

# 6. Parecer Técnico

## Conclusão

O presente catálogo documentou as três fontes oficiais de dados utilizadas no projeto:

- **Feature Table `ml_aluno`**: base principal da modelagem supervisionada;
- **Censo Escolar 2023**: fonte complementar de informações educacionais e estruturais das escolas;
- **INSE 2023**: fonte complementar de indicadores de nível socioeconômico em nível escolar.

As variáveis foram catalogadas quanto ao tipo de dado, descrição funcional e categoria, estabelecendo a documentação necessária para as próximas etapas de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

Este notebook tem caráter documental e não realiza transformações ou integrações entre as bases, preservando a rastreabilidade dos dados originais.